# MNIST Demo: Image Classification with CREST

This notebook walks through a complete CREST workflow on the MNIST handwritten-digit
classification task. It covers the typical lifecycle of a CREST model:

1. **Loading data** via `DataServer` (`crest/data_server/DataServer.py`)
2. **Building a CNN** — two `Node`s composed in a `HierarchalTensorGraph`
3. **Compiling and training** with `Model.compile` / `Model.fit`
4. **Evaluating** with `Model.evaluate` and CREST `Metrics`
5. **Predictions** — via `Batcher`, raw dict input, and with extra output coordinates
6. **Saving and reloading** a trained model with `Model.save` / `Model.load`

In [ ]:
# Allow crest to be imported from the examples/ sub-directory
%cd -q ..

## 1. Loading MNIST Data

`DataServer.load('mnist')` (`crest/data_server/DataServer.py:load_mnist`) downloads
four pre-built `StructuredDataset` JSON files from the CREST data server and returns
them as `[train, valid, test, pred]`. On subsequent calls the files are read from the
local cache.

Each dataset stores its arrays in `ds.data` (a tuple) and their names in `ds.labels`.
For the MNIST datasets `labels = ['image', 'class']`; the prediction set has only
`labels = ['image']` (no ground-truth labels).

In [ ]:
import numpy as np
import tensorflow as tf

from crest import DataServer
from crest.data.loading import StructuredDataset

ds_train, ds_valid, ds_test, ds_pred = DataServer.load('mnist')

In [ ]:
[INP, OUT] = ds_train.labels   # 'image', 'class'
n_classes  = 10
patch_size = 28

print(f"Labels : {ds_train.labels}")
print(f"Train  : images {ds_train.data[0].shape}  dtype {ds_train.data[0].dtype}")

In [ ]:
# Raw images are int64 in [0, 255]; normalize to float32 in [0, 1] because
# TensorSpec defaults to dtype='float32'
def _norm(arr):
    return arr.astype('float32') / 255.0

ds_train = StructuredDataset(_norm(ds_train.data[0]), ds_train.data[1], labels=ds_train.labels)
ds_valid = StructuredDataset(_norm(ds_valid.data[0]), ds_valid.data[1], labels=ds_valid.labels)
ds_test  = StructuredDataset(_norm(ds_test.data[0]),  ds_test.data[1],  labels=ds_test.labels)
ds_pred  = StructuredDataset(_norm(ds_pred.data[0]),                    labels=ds_pred.labels)

print(f"After normalization:")
print(f"Train  : images {ds_train.data[0].shape}  dtype {ds_train.data[0].dtype}  "
      f"range [{ds_train.data[0].min():.2f}, {ds_train.data[0].max():.2f}]")
print(f"Valid  : images {ds_valid.data[0].shape}  labels {ds_valid.data[1].shape}")
print(f"Test   : images {ds_test.data[0].shape}   labels {ds_test.data[1].shape}")
print(f"Pred   : images {ds_pred.data[0].shape}")

## 2. Building a CNN as a CREST HTG

A `HierarchalTensorGraph` (HTG) composes one or more `Node`s into a coupled system.
Each `Node` (`crest/model/Node.py`) wraps a Python callable and declares its tensor
interface via `TensorSpec` (`crest/model/TensorSpec.py`). `TensorSpec` takes a plain
list of dimensions as its first positional argument (e.g. `TensorSpec([None, 28, 28])`).

Here we use two nodes chained together:

- **preprocessing** — adds the channel dimension required by Conv2D
- **cnn** — a Keras `Sequential` stack (Conv2D → Flatten → Dropout → Dense softmax)

The intermediate tensor uses the key `'x'` so input/output names stay unambiguous.
`add_edge` wires the nodes; since both share key `'x'` on the shared edge, no `rename`
is needed.

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Reshape, Conv2D, Flatten, Dropout, Dense

from crest import Node, HierarchalTensorGraph
from crest.model.TensorSpec import TensorSpec

INTER = 'x'   # Intermediate key between preprocessing and CNN nodes

# Node 1: preprocessing
preprocess_seq = Sequential([Reshape((patch_size, patch_size, 1))])
preprocessing = Node(
    node=lambda X: {INTER: preprocess_seq(X[INP])},
    name='preprocessing',
    inputs={INP:    TensorSpec([None, patch_size, patch_size])},
    outputs={INTER: TensorSpec([None, patch_size, patch_size, 1])},
)

# Node 2: CNN — Conv2D → Flatten → Dropout → Dense softmax
cnn_layers = Sequential([
    Conv2D(10, 3, padding='same', activation='relu'),
    Flatten(),
    Dropout(0.3),
    Dense(n_classes, activation='softmax'),
])
cnn_node = Node(
    node=lambda X: {OUT: cnn_layers(X[INTER])},
    name='cnn',
    inputs={INTER: TensorSpec([None, patch_size, patch_size, 1])},
    outputs={OUT:  TensorSpec([None, n_classes])},
)

In [ ]:
graph = HierarchalTensorGraph(
    name='mnist',
    inputs={INP: TensorSpec([None, patch_size, patch_size])},
    outputs={OUT: TensorSpec([None, n_classes])},
)
graph.add_edge('input',       preprocessing)
graph.add_edge(preprocessing, cnn_node)
graph.add_edge(cnn_node,      'output')

graph.draw()

## 3. Compile and Train

`Model` (`crest/model/Model.py`) wraps the HTG in a Keras functional model and exposes
`compile` / `fit` / `evaluate` / `predict` methods.

- `model.compile(**kwargs)` — forwards `kwargs` to `keras.Model.compile`; standard Keras
  losses and optimizers are supported.
- `model.metrics` — the CREST `Metrics` registry (`crest/utils/Metrics.py`) attached at
  construction time. Custom Python callables registered here are serialized with the
  model on `save`.
- `model.fit(batcher, ...)` — when given a `Batcher`, `fit` manages the context manager
  internally (`with data as batcher: self.model.fit(iter(batcher), ...)`), so no outer
  `with` block is needed.

The `steps_per_epoch` value is set for a quick demo run. Full-accuracy training would 
use `steps_per_epoch=250`.

In [ ]:
from crest.model.Model import Model
from crest.data.batching.Batcher import Batcher
from crest.utils.Metrics import Metrics

model = Model(graph)

# Register a custom CREST metric before compiling
def squared_diff(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

model.metrics.register('squared_diff', squared_diff)
print("Custom CREST metrics:", model.metrics.customs)

In [ ]:
# Compile with loss only — omit metrics= to avoid a TF/Keras crash
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
)

In [ ]:
batch_size = 200

batch_train = Batcher(ds_train, **{
    'batch_size': batch_size,
    'features'  : [[INP], [OUT]],  # inputs / targets
    'repeat'    : True,
    'workers'   : 0,
    'prefetch'  : False,
})

In [ ]:
# model.fit opens its own context manager for batch_train
model.fit(batch_train, **{
    'steps_per_epoch': 50,
    'epochs'         : 3,
    'verbose'        : 1,
})
batch_train.close()

## 4. Evaluate

`model.evaluate` accepts a `Batcher` directly.
It wraps it as a plain Python iterator via `iter(batcher)` and forwards it to
`keras.Model.evaluate`.

Because `metrics=` is omitted from `compile`, `evaluate` returns only the
loss. Accuracy is computed manually from the raw model predictions: call
`model.predict` on the test set, take `np.argmax` over the class axis, and compare
with the ground-truth integer labels.

In [ ]:
n_test       = len(ds_test.data[0])           # 10 000 test samples
n_steps_eval = n_test // batch_size           # 10 000 // 200 = 50

batch_test = Batcher(ds_test, **{
    'batch_size': batch_size,
    'features'  : [[INP], [OUT]],
    'repeat'    : False,
    'shuffle'   : False,
    'workers'   : 0,
    'prefetch'  : False,
})
# model.evaluate manages the Batcher context internally
results = model.evaluate(batch_test, steps=n_steps_eval, verbose=0, return_dict=True)
print("Test loss:", results['loss'])

In [ ]:
# Compute accuracy manually 
batch_test2 = Batcher(ds_test, **{
    'batch_size': batch_size,
    'features'  : [INP],
    'repeat'    : False,
    'shuffle'   : False,
    'workers'   : 0,
    'prefetch'  : False,
})
preds_test = model.predict(batch_test2)
batch_test2.close()

y_pred_classes = np.argmax(preds_test[OUT], axis=-1)
y_true_classes = ds_test.data[1]   # integer labels from DataServer
test_accuracy  = np.mean(y_pred_classes == y_true_classes)
print(f"Test accuracy : {test_accuracy:.4f}  "
      f"({int(test_accuracy * n_test)}/{n_test} correct)")

## 5. Predictions

`Model.predict` supports three calling styles:

- **Batcher** — CREST manages the context and concatenates batches internally;
  with `repeat=False` the batcher exhausts naturally (no `steps` needed here because
  `predict` uses its own loop rather than Keras's `EpochIterator`).
- **Raw dict** — pass `{input_name: numpy_array}` directly; useful for small ad-hoc
  inference without constructing a Batcher.
- Batcher with `coords` — extra keys (e.g. spatial coordinates or timestamps) are
  carried through the prediction step and returned alongside the model output.
  `coords` requires a `Batcher`; passing a plain dataset raises `ValueError`.

In [ ]:
# Predict with a Batcher
# repeat=False exhausts the dataset in one pass; no steps kwarg required.
batch_pred = Batcher(ds_pred, **{
    'batch_size': batch_size,
    'features'  : [INP],     # prediction-only set; no target column
    'repeat'    : False,
    'shuffle'   : False,
    'workers'   : 0,
    'prefetch'  : False,
})

preds = model.predict(batch_pred)
batch_pred.close()

predicted_classes = np.argmax(preds[OUT], axis=-1)
print(f"Predictions shape : {preds[OUT].shape}")
print(f"First 10 predicted: {predicted_classes[:10]}")
print(f"Class distribution: {np.bincount(predicted_classes)}")

In [ ]:
# Predict with a raw dict
# model.predict delegates to keras.Model.predict for non-Batcher inputs.
x_sample = ds_test.data[0][:10]   # first 10 test images, shape (10, 28, 28)

preds_dict = model.predict({INP: x_sample})

print(f"Output keys : {list(preds_dict.keys())}")
print(f"Predicted   : {np.argmax(preds_dict[OUT], axis=-1)}")
print(f"True labels : {ds_test.data[1][:10]}")

In [ ]:
# Predict with extra coordinates
# Attach synthetic lat/lon metadata to each prediction sample.
# coords must be passed as a Batcher; using a plain dataset raises ValueError.
from crest.data.loading import StructuredDataset

x   = ds_pred.data[0]
lat = np.arange(len(x), dtype=float)
lon = np.arange(len(x), dtype=float)

ds_aux    = StructuredDataset(lat, lon, x, labels=['lat', 'lon', INP])
batch_aux = Batcher(ds_aux, **{
    'batch_size': batch_size,
    'features'  : ['lat', 'lon', INP],
    'repeat'    : False,
    'shuffle'   : False,
    'workers'   : 0,
    'prefetch'  : False,
})

preds_with_coords = model.predict(batch_aux, coords=['lat', 'lon'])
batch_aux.close()

print("Keys in output:", list(preds_with_coords.keys()))
print("lat (first 5): ", preds_with_coords['lat'][:5])
print("lon (first 5): ", preds_with_coords['lon'][:5])

## 6. Save and Reload

`Model.save(filepath)` writes a zip archive containing:

- **graph.json** — the HTG encoded with `dill` and base64, plus the registry name used
  to reconstruct the class on load (`HierarchalTensorGraph.encode`)
- **model.weights.h5** — Keras trainable weights
- **metrics.json** — the CREST `Metrics` registry (including custom metrics) via
  `Metrics.to_json` / `jsonpickle`
- **compile_args.json** — Keras compile configuration

`Model.load(filepath)` is a `classmethod` that reverses the process: it looks up the
HTG class by name in HTG registry, decodes the graph, restores weights, and re-compiles. 

In [ ]:
import os

model.save('mnist.crest')
loaded_model = Model.load('mnist.crest')

# Verify loaded model gives identical predictions
pred_orig   = model.predict({INP: x_sample})[OUT]
pred_loaded = loaded_model.predict({INP: x_sample})[OUT]
max_diff    = np.abs(pred_orig - pred_loaded).max()

print(f"Max prediction difference after reload: {max_diff:.2e}")
print("Custom metrics preserved:", loaded_model.metrics.customs)

os.remove('mnist.crest')